[코랩(Colab)으로 열기](https://github.com/dhnam-data/Datafile/blob/main/StatisticsHealthCare/code/13_4_%EC%9D%BC%EC%9B%90%EB%B0%B0%EC%B9%98_%EB%B6%84%EC%82%B0%EB%B6%84%EC%84%9D.ipynb)

귀무가설(H0): 금연계획 집단의 하루 평균 흡연량은 모두 같다. [H0: μ1 ＝ μ2 ＝ μ3 ＝ μ4]

대립가설(H1): 적어도 한 집단의 하루 평균 흡연량은 다른 집단과 다르다. [H1: μ1 ≠ μ2 ≠ μ3 ≠ μ4]

In [2]:
import pandas as pd
pd.set_option('display.max_columns', None) # 컬럼 전체보기

# 파일 불러오기
df = pd.read_excel("https://github.com/dhnam-data/Datafile/raw/refs/heads/main/StatisticsHealthCare/data/2.%ED%8F%89%EA%B7%A0%EB%B9%84%EA%B5%902.xlsx")
print(df.shape)
df.head()

(1938, 3)


,p2005_3aq5,p2005_3aq6,p2005_3aq8
0,1,3,4
1,1,5,3
2,1,20,4
3,1,10,3
4,1,5,3


# Levene의 등분산 검정

In [3]:
from statsmodels.stats.oneway import test_scale_oneway

a = df.loc[df['p2005_3aq8'] == 1, 'p2005_3aq6']
b = df.loc[df['p2005_3aq8'] == 2, 'p2005_3aq6']
c = df.loc[df['p2005_3aq8'] == 3, 'p2005_3aq6']
d = df.loc[df['p2005_3aq8'] == 4, 'p2005_3aq6']

result = test_scale_oneway(
    [a, b, c, d],
    method='equal',
    center='mean',
    transform='abs'
)

print('Levene 통계량:', result.statistic)
print('p값:', result.pvalue)

Levene 통계량: 2.663887437696832
p값: 0.04648408502092368


# 일원배치 분산분석

In [4]:
# p2005_3aq6: 금연계획
# p2005_3aq8: 하루 평균 흡연량

# 결측치 제거
df = df.dropna(subset=['p2005_3aq8', 'p2005_3aq6'])


a = df[df['p2005_3aq8'] == 1]
b = df[df['p2005_3aq8'] == 2]
c = df[df['p2005_3aq8'] == 3]
d = df[df['p2005_3aq8'] == 4]


from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

model = ols('p2005_3aq6 ~ C(p2005_3aq8)', df).fit()
result = anova_lm(model)
result

,count
p2005_3aq8,
4,1215
3,588
2,92
1,43


,df,sum_sq,mean_sq,F,PR(>F)
C(p2005_3aq8),3.0,2474.777825,824.925942,14.312037,3.177268e-09
Residual,1934.0,111473.072020,57.638610,NaN,NaN


# 사후검정

## 등분산

In [5]:
!pip install scikit-posthocs -q
import scikit_posthocs as sp

result = sp.posthoc_scheffe(
    df,
    val_col='p2005_3aq6',    # 종속변수
    group_col='p2005_3aq8'   # 집단변수
)

display(result)

,4,3,2,1
4,1.000000,0.000031,0.002742,0.002986
3,0.000031,1.000000,0.551312,0.210751
2,0.002742,0.551312,1.000000,0.830218
1,0.002986,0.210751,0.830218,1.000000


## 이분산

In [6]:
!pip install scikit-posthocs -q
import scikit_posthocs as sp

result = sp.posthoc_dunnett_t3(
    df,
    val_col='p2005_3aq6',    # 종속변수
    group_col='p2005_3aq8'  # 집단변수
)

display(result)


,4,3,2,1
4,1.000000,0.000004,0.001350,0.000088
3,0.000004,1.000000,0.590328,0.049499
2,0.001350,0.590328,1.000000,0.839659
1,0.000088,0.049499,0.839659,1.000000


# Levene의 등분산 검정

In [19]:
import pingouin as pg

df = df.dropna(subset=['p2005_3aq6', 'p2005_3aq8'])

result = pg.homoscedasticity(
    data=df,
    dv='p2005_3aq6',        # 종속변수
    group='p2005_3aq8',     # 집단변수
    method='levene',
    center='mean'          # 평균 기준 Levene 검정
)

display(result)

,W,pval,equal_var
levene,2.663887,0.046484,False


# 일원배치 분산분석

In [14]:
display(pg.anova(data=df, dv='p2005_3aq6', between='p2005_3aq8', ss_type=3, detailed=True, effsize='np2'))

,Source,SS,DF,MS,F,p_unc,np2
0,p2005_3aq8,2474.777825,3,824.925942,14.312037,3.177268e-09,0.021719
1,Within,111473.072020,1934,57.638610,NaN,NaN,NaN


# 사후검정

## 등분산

In [16]:
!pip install scikit-posthocs -q
import scikit_posthocs as sp

result = sp.posthoc_scheffe(
    df,
    val_col='p2005_3aq6',    # 종속변수
    group_col='p2005_3aq8'   # 집단변수
)

display(result)

,4,3,2,1
4,1.000000,0.000031,0.002742,0.002986
3,0.000031,1.000000,0.551312,0.210751
2,0.002742,0.551312,1.000000,0.830218
1,0.002986,0.210751,0.830218,1.000000


## 이분산

In [17]:
!pip install scikit-posthocs -q
import scikit_posthocs as sp

result = sp.posthoc_dunnett_t3(
    df,
    val_col='p2005_3aq6',    # 종속변수
    group_col='p2005_3aq8'  # 집단변수
)

display(result)


,4,3,2,1
4,1.000000,0.000004,0.001350,0.000088
3,0.000004,1.000000,0.590328,0.049499
2,0.001350,0.590328,1.000000,0.839659
1,0.000088,0.049499,0.839659,1.000000
